# Tests de la PGD (x, μ) à P = 2200

Expérience : `Exp-pgd-mu-1.md` (vault). Notebook `tests-…` : il lance les runs (V0 … V5 du « Recentrage ») et enregistre chacun dans `results/pgd/<run>/` (`config.json`, `decomposition.pt`, `history.csv`, `stages.csv`, `error.csv`, `signs.csv`, `diagram.csv`).

- w(x, μ) = Σ_m w_m(x) λ_m(μ), μ = k^{1/4} (k : raideur de la fondation) ; w_m : monôme en x, λ_m : monôme en μ.
- e(n) : erreur relative L² de la PGD tronquée à n modes contre la référence EF, estimée sur les 25 points LHS (x_i, μ_i) ; s_i = ±1 aligne le signe de la référence sur la PGD.
- λ_min : plus petite valeur propre de la hessienne de Π en w_PGD(·, μ) (problème non décomposé, k = μ⁴) ; λ_min ≤ 0 : col.

Lit `results/reference/lhs_N25_seed0.pt` (`reference_mu.py`) et `../kolmogorov_study/results/c1/branches.csv`, `critical.csv` (`branches_k.py`). Aucune figure de ce notebook n'est dans la note pour l'instant.

In [ ]:
import json
import sys
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "../..")
sys.path.insert(0, "../kolmogorov_study")
from neurom.quadratures import FourPoints1D
from neurom.training import (
    AnyOf, MaxStages, Phase, ProgressLog, SimultaneousTrainer, Stagnation,
    Usefulness,
)

from main_pgd_mu import (
    main, save_decomposition, load_decomposition, sample_monoms, axial_displacement,
    reference_values, align_signs, errors, lambda_min,
)
from plotting import plot_branches_k, plot_loss_history, plot_monoms, plot_deflection_slider
from pod import mass_matrix
from utils import rescale

RESULTS = Path("results")
REFERENCE = RESULTS / "reference" / "lhs_N25_seed0.pt"
C1 = Path("../kolmogorov_study/results/c1")
PHYS = dict(E=210e9, A=1e-4, I=1e-8 / 12, length=1.0)  # steel, 1 cm x 1 cm: for `rescale` only
QUAD = FourPoints1D()

## Run 1 - 20 elems

Paramètres du « Choix initial » de l'Exp (V0). Pour un autre run : changer `RUN` et un seul facteur.

In [ ]:
RUN = "V0"
CONFIG = dict(
    P=2200.0, n_elements=20, n_mu=20, mu_range=(0.0, 40.0), n_modes=15,
    noise_amplitude=1e-2, seed=0,
    # Algorithm 1 of FENNI-PGD (values of Training_NeuROM on Develop_Daby)
    adam_lr=1e-2, adam_max_iter=5000, eta_c=1e-4, patience=10, no_improvement=None,
    min_useful=15,
    # final LBFGS on the whole decomposition (§3.2.2)
    lbfgs_lr=1.0, final_lbfgs_max_iter=500, tol=1e-8,
)
RECOMPUTE = True  # False: reload results/pgd/<RUN>/ when it exists
PROGRESS_EVERY = {"adam": 500, "lbfgs": 0.1}  # adam: every 500 it; lbfgs: every 10 % of its max_iter

In [ ]:
out = RESULTS / "pgd" / RUN
if RECOMPUTE or not (out / "decomposition.pt").exists():
    out.mkdir(parents=True, exist_ok=True)
    c = CONFIG
    adam = Phase(
        lambda params: torch.optim.Adam(params, lr=c["adam_lr"]),
        Stagnation(tol=c["eta_c"], patience=c["patience"],
                   no_improvement=c["no_improvement"], max_iter=c["adam_max_iter"]),
        "adam",
    )
    lbfgs = Phase(
        lambda params: torch.optim.LBFGS(params, lr=c["lbfgs_lr"], line_search_fn="strong_wolfe"),
        Stagnation(tol=c["tol"], patience=0, no_improvement=None,
                   max_iter=c["final_lbfgs_max_iter"]),
        "lbfgs",
    )
    results = main(
        n_elements=c["n_elements"], n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"],
        noise_amplitude=c["noise_amplitude"], seed=c["seed"], phases=[adam],
        enrichment_criterion=AnyOf(Usefulness(tol=c["eta_c"], min_useful=c["min_useful"]),
                                   MaxStages(c["n_modes"])),
        quadrature=QUAD, **PHYS, trainer=partial(SimultaneousTrainer, final_phases=[lbfgs]),
        progress=ProgressLog(every=PROGRESS_EVERY),
    )
    cppgd, history, problem = results["cppgd"], results["history"], results["problem"]
    records = history.stages + ([history.final] if history.final is not None else [])
    save_decomposition(cppgd, out / "decomposition.pt", n_elements=c["n_elements"],
                       n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"])
    (out / "config.json").write_text(json.dumps(CONFIG, indent=2))
    pd.DataFrame({
        "loss": history.losses,
        "mode": [r.stage for r in records for _ in r.losses],  # final LBFGS: stage = number of modes
        "phase": [p for r in records for p in r.phases],
    }).to_csv(out / "history.csv", index=False)
    pd.DataFrame([
        {"mode": r.stage, "n_iter": r.n_iter, "stop_reason": r.stop_reason, "energy": r.energy,
         "final": r is history.final}
        for r in records
    ]).to_csv(out / "stages.csv", index=False)
else:
    cppgd, _, problem = load_decomposition(out / "decomposition.pt", quadrature=QUAD)
    CONFIG = json.loads((out / "config.json").read_text())

history = pd.read_csv(out / "history.csv")
stages = pd.read_csv(out / "stages.csv")
stages

### Erreur e(n) sur les points LHS

Paragraphe « Erreur » de l'Exp ; s_i par q_PGDᵀ M q_EF (même maillage) ou grille fine (maillages différents).

In [ ]:
ref = torch.load(REFERENCE)
assert ref["P"] == CONFIG["P"] and tuple(ref["mu_box"]) == tuple(CONFIG["mu_range"]), "reference at another P or box"
if ref["n_elements"] != CONFIG["n_elements"]:
    print(f"warning: reference on {ref['n_elements']} elements in x, PGD on {CONFIG['n_elements']}")

s = align_signs(cppgd, ref["Q"], ref["mu"], ref["n_elements"], CONFIG["P"], M=mass_matrix(ref["n_elements"]))
w_ref = reference_values(ref["Q"], ref["n_elements"], CONFIG["P"], ref["x"], per_solution=True)
e = errors(cppgd, ref["x"], ref["mu"], w_ref, s)

error = pd.DataFrame({"n": range(len(e)), "e": e.tolist()})
error.to_csv(out / "error.csv", index=False)
pd.DataFrame({"x": ref["x"], "mu": ref["mu"], "s": s}).to_csv(out / "signs.csv", index=False)
print(f"s_i = -1 at {int((s < 0).sum())} of {len(s)} points")
error.set_index("n").T

### Diagramme de bifurcation w(1/4) – μ

w_PGD(1/4, μ) sur 201 μ, avec λ_min en chaque μ (croix rouges : cols), sur les branches exactes de P = 2200 (C1, 100 éléments ; μ_cr de C1).

In [ ]:
mu_grid = torch.linspace(*CONFIG["mu_range"], 201)
x_q, mu_m = sample_monoms(cppgd, torch.tensor([0.25]), mu_grid)
lam = lambda_min(cppgd, mu_grid, CONFIG["P"], CONFIG["n_elements"], quadrature=QUAD)
diagram = pd.DataFrame({"mu": mu_grid, "w_quarter": (x_q * mu_m).sum(dim=0), "lambda_min": lam})
diagram.to_csv(out / "diagram.csv", index=False)
print(f"lambda_min <= 0 at {int((diagram.lambda_min <= 0).sum())} of {len(diagram)} mu")

In [ ]:
branches = pd.read_csv(C1 / "branches.csv")
mu_cr = pd.read_csv(C1 / "critical.csv").set_index("P").mu_cr[CONFIG["P"]]
plot_branches_k(
    branches, None, CONFIG["P"], mu_critical=mu_cr, series={f"PGD {RUN}": diagram},
    title=f"PGD {RUN}, P = {CONFIG['P']:g}: w(1/4) against mu (red crosses: lambda_min <= 0)",
).show()

### Exploration

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogy(error.n, error.e, "o-")
ax.set(xlabel="n (modes)", ylabel="e(n)", title=f"PGD {RUN}: relative L2 error on the LHS points")
ax.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
plot_loss_history(history.loss.tolist(), history["mode"].tolist()).show()

In [ ]:
x_fine = torch.linspace(0.0, 1.0, 201)
x_m, mu_m = sample_monoms(cppgd, x_fine, mu_grid)
plot_monoms(x_fine, x_m, mu_grid, mu_m).show()  # right panel: mu monoms (axis titled P by plot_monoms)

In [ ]:
# Deformed beam (x_hat + u_hat, w_hat): u at the nodes, linearly interpolated in x as in `main`
x_nodes = problem["x_nodes"]
u_nodes = axial_displacement(problem["field_layout"], cppgd, x_nodes, mu_m, CONFIG["P"])
u_fine = torch.from_numpy(np.stack(
    [np.interp(x_fine.numpy(), x_nodes.numpy(), u) for u in u_nodes.T.numpy()], axis=1,
))
x_hat, w_hat, u_hat, _ = rescale(x_fine, cppgd.assemble([x_fine, mu_grid]).detach(), u_fine,
                                 CONFIG["P"], **PHYS)
plot_deflection_slider(x_hat[:, None] + u_hat, mu_grid, w_hat,
                       title=f"PGD {RUN}: deformed beam (x + u, w) [m]").show()

## Run 2 - 50 elems

Paramètres du « Choix initial » de l'Exp (V0). Pour un autre run : changer `RUN` et un seul facteur.

In [ ]:
RUN = "V0"
CONFIG = dict(
    P=2200.0, n_elements=20, n_mu=50, mu_range=(0.0, 40.0), n_modes=15,
    noise_amplitude=1e-2, seed=0,
    # Algorithm 1 of FENNI-PGD (values of Training_NeuROM on Develop_Daby)
    adam_lr=1e-2, adam_max_iter=5000, eta_c=1e-4, patience=10, no_improvement=None,
    min_useful=15,
    # final LBFGS on the whole decomposition (§3.2.2)
    lbfgs_lr=1.0, final_lbfgs_max_iter=500, tol=1e-6,
)
RECOMPUTE = True  # False: reload results/pgd/<RUN>/ when it exists
PROGRESS_EVERY = {"adam": 500, "lbfgs": 0.1}  # adam: every 500 it; lbfgs: every 10 % of its max_iter

In [ ]:
out = RESULTS / "pgd" / RUN
if RECOMPUTE or not (out / "decomposition.pt").exists():
    out.mkdir(parents=True, exist_ok=True)
    c = CONFIG
    adam = Phase(
        lambda params: torch.optim.Adam(params, lr=c["adam_lr"]),
        Stagnation(tol=c["eta_c"], patience=c["patience"],
                   no_improvement=c["no_improvement"], max_iter=c["adam_max_iter"]),
        "adam",
    )
    lbfgs = Phase(
        lambda params: torch.optim.LBFGS(params, lr=c["lbfgs_lr"], line_search_fn="strong_wolfe"),
        Stagnation(tol=c["tol"], patience=0, no_improvement=None,
                   max_iter=c["final_lbfgs_max_iter"]),
        "lbfgs",
    )
    results = main(
        n_elements=c["n_elements"], n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"],
        noise_amplitude=c["noise_amplitude"], seed=c["seed"], phases=[adam],
        enrichment_criterion=AnyOf(Usefulness(tol=c["eta_c"], min_useful=c["min_useful"]),
                                   MaxStages(c["n_modes"])),
        quadrature=QUAD, **PHYS, trainer=partial(SimultaneousTrainer, final_phases=[lbfgs]),
        progress=ProgressLog(every=PROGRESS_EVERY),
    )
    cppgd, history, problem = results["cppgd"], results["history"], results["problem"]
    records = history.stages + ([history.final] if history.final is not None else [])
    save_decomposition(cppgd, out / "decomposition.pt", n_elements=c["n_elements"],
                       n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"])
    (out / "config.json").write_text(json.dumps(CONFIG, indent=2))
    pd.DataFrame({
        "loss": history.losses,
        "mode": [r.stage for r in records for _ in r.losses],  # final LBFGS: stage = number of modes
        "phase": [p for r in records for p in r.phases],
    }).to_csv(out / "history.csv", index=False)
    pd.DataFrame([
        {"mode": r.stage, "n_iter": r.n_iter, "stop_reason": r.stop_reason, "energy": r.energy,
         "final": r is history.final}
        for r in records
    ]).to_csv(out / "stages.csv", index=False)
else:
    cppgd, _, problem = load_decomposition(out / "decomposition.pt", quadrature=QUAD)
    CONFIG = json.loads((out / "config.json").read_text())

history = pd.read_csv(out / "history.csv")
stages = pd.read_csv(out / "stages.csv")
stages

### Erreur e(n) sur les points LHS

Paragraphe « Erreur » de l'Exp ; s_i par q_PGDᵀ M q_EF (même maillage) ou grille fine (maillages différents).

In [ ]:
ref = torch.load(REFERENCE)
assert ref["P"] == CONFIG["P"] and tuple(ref["mu_box"]) == tuple(CONFIG["mu_range"]), "reference at another P or box"
if ref["n_elements"] != CONFIG["n_elements"]:
    print(f"warning: reference on {ref['n_elements']} elements in x, PGD on {CONFIG['n_elements']}")

s = align_signs(cppgd, ref["Q"], ref["mu"], ref["n_elements"], CONFIG["P"], M=mass_matrix(ref["n_elements"]))
w_ref = reference_values(ref["Q"], ref["n_elements"], CONFIG["P"], ref["x"], per_solution=True)
e = errors(cppgd, ref["x"], ref["mu"], w_ref, s)

error = pd.DataFrame({"n": range(len(e)), "e": e.tolist()})
error.to_csv(out / "error.csv", index=False)
pd.DataFrame({"x": ref["x"], "mu": ref["mu"], "s": s}).to_csv(out / "signs.csv", index=False)
print(f"s_i = -1 at {int((s < 0).sum())} of {len(s)} points")
error.set_index("n").T

### Diagramme de bifurcation w(1/4) – μ

w_PGD(1/4, μ) sur 201 μ, avec λ_min en chaque μ (croix rouges : cols), sur les branches exactes de P = 2200 (C1, 100 éléments ; μ_cr de C1).

In [ ]:
mu_grid = torch.linspace(*CONFIG["mu_range"], 201)
x_q, mu_m = sample_monoms(cppgd, torch.tensor([0.25]), mu_grid)
lam = lambda_min(cppgd, mu_grid, CONFIG["P"], CONFIG["n_elements"], quadrature=QUAD)
diagram = pd.DataFrame({"mu": mu_grid, "w_quarter": (x_q * mu_m).sum(dim=0), "lambda_min": lam})
diagram.to_csv(out / "diagram.csv", index=False)
print(f"lambda_min <= 0 at {int((diagram.lambda_min <= 0).sum())} of {len(diagram)} mu")

In [ ]:
branches = pd.read_csv(C1 / "branches.csv")
mu_cr = pd.read_csv(C1 / "critical.csv").set_index("P").mu_cr[CONFIG["P"]]
plot_branches_k(
    branches, None, CONFIG["P"], mu_critical=mu_cr, series={f"PGD {RUN}": diagram},
    title=f"PGD {RUN}, P = {CONFIG['P']:g}: w(1/4) against mu (red crosses: lambda_min <= 0)",
).show()

### Exploration

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogy(error.n, error.e, "o-")
ax.set(xlabel="n (modes)", ylabel="e(n)", title=f"PGD {RUN}: relative L2 error on the LHS points")
ax.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
plot_loss_history(history.loss.tolist(), history["mode"].tolist()).show()

In [ ]:
x_fine = torch.linspace(0.0, 1.0, 201)
x_m, mu_m = sample_monoms(cppgd, x_fine, mu_grid)
plot_monoms(x_fine, x_m, mu_grid, mu_m).show()  # right panel: mu monoms (axis titled P by plot_monoms)

In [ ]:
# Deformed beam (x_hat + u_hat, w_hat): u at the nodes, linearly interpolated in x as in `main`
x_nodes = problem["x_nodes"]
u_nodes = axial_displacement(problem["field_layout"], cppgd, x_nodes, mu_m, CONFIG["P"])
u_fine = torch.from_numpy(np.stack(
    [np.interp(x_fine.numpy(), x_nodes.numpy(), u) for u in u_nodes.T.numpy()], axis=1,
))
x_hat, w_hat, u_hat, _ = rescale(x_fine, cppgd.assemble([x_fine, mu_grid]).detach(), u_fine,
                                 CONFIG["P"], **PHYS)
plot_deflection_slider(x_hat[:, None] + u_hat, mu_grid, w_hat,
                       title=f"PGD {RUN}: deformed beam (x + u, w) [m]").show()

## Run 3 - 20 elems Adam pure

Paramètres du « Choix initial » de l'Exp (V0). Pour un autre run : changer `RUN` et un seul facteur.

In [ ]:
RUN = "V2"
CONFIG = dict(
    P=2200.0, n_elements=20, n_mu=20, mu_range=(0.0, 40.0), n_modes=15,
    noise_amplitude=1e-2, seed=0,
    # Algorithm 1 of FENNI-PGD (values of Training_NeuROM on Develop_Daby)
    adam_lr=1e-2, adam_max_iter=5000, eta_c=1e-4, patience=10, no_improvement=None,
    min_useful=15,
    # final LBFGS on the whole decomposition (§3.2.2)
    lbfgs_lr=1.0, final_lbfgs_max_iter=500, tol=1e-8,
)
RECOMPUTE = True  # False: reload results/pgd/<RUN>/ when it exists
PROGRESS_EVERY = {"adam": 500, "lbfgs": 0.1}  # adam: every 500 it; lbfgs: every 10 % of its max_iter

In [ ]:
out = RESULTS / "pgd" / RUN
if RECOMPUTE or not (out / "decomposition.pt").exists():
    out.mkdir(parents=True, exist_ok=True)
    c = CONFIG
    adam = Phase(
        lambda params: torch.optim.Adam(params, lr=c["adam_lr"]),
        Stagnation(tol=c["eta_c"], patience=c["patience"],
                   no_improvement=c["no_improvement"], max_iter=c["adam_max_iter"]),
        "adam",
    )
    lbfgs = Phase(
        lambda params: torch.optim.LBFGS(params, lr=c["lbfgs_lr"], line_search_fn="strong_wolfe"),
        Stagnation(tol=c["tol"], patience=0, no_improvement=None,
                   max_iter=c["final_lbfgs_max_iter"]),
        "lbfgs",
    )
    results = main(
        n_elements=c["n_elements"], n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"],
        noise_amplitude=c["noise_amplitude"], seed=c["seed"], phases=[adam],
        enrichment_criterion=AnyOf(Usefulness(tol=c["eta_c"], min_useful=c["min_useful"]),
                                   MaxStages(c["n_modes"])),
        quadrature=QUAD, **PHYS, trainer=partial(SimultaneousTrainer, final_phases=[]),
        progress=ProgressLog(every=PROGRESS_EVERY),
    )
    cppgd, history, problem = results["cppgd"], results["history"], results["problem"]
    records = history.stages + ([history.final] if history.final is not None else [])
    save_decomposition(cppgd, out / "decomposition.pt", n_elements=c["n_elements"],
                       n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"])
    (out / "config.json").write_text(json.dumps(CONFIG, indent=2))
    pd.DataFrame({
        "loss": history.losses,
        "mode": [r.stage for r in records for _ in r.losses],  # final LBFGS: stage = number of modes
        "phase": [p for r in records for p in r.phases],
    }).to_csv(out / "history.csv", index=False)
    pd.DataFrame([
        {"mode": r.stage, "n_iter": r.n_iter, "stop_reason": r.stop_reason, "energy": r.energy,
         "final": r is history.final}
        for r in records
    ]).to_csv(out / "stages.csv", index=False)
else:
    cppgd, _, problem = load_decomposition(out / "decomposition.pt", quadrature=QUAD)
    CONFIG = json.loads((out / "config.json").read_text())

history = pd.read_csv(out / "history.csv")
stages = pd.read_csv(out / "stages.csv")
stages

### Erreur e(n) sur les points LHS

Paragraphe « Erreur » de l'Exp ; s_i par q_PGDᵀ M q_EF (même maillage) ou grille fine (maillages différents).

In [ ]:
ref = torch.load(REFERENCE)
assert ref["P"] == CONFIG["P"] and tuple(ref["mu_box"]) == tuple(CONFIG["mu_range"]), "reference at another P or box"
if ref["n_elements"] != CONFIG["n_elements"]:
    print(f"warning: reference on {ref['n_elements']} elements in x, PGD on {CONFIG['n_elements']}")

s = align_signs(cppgd, ref["Q"], ref["mu"], ref["n_elements"], CONFIG["P"], M=mass_matrix(ref["n_elements"]))
w_ref = reference_values(ref["Q"], ref["n_elements"], CONFIG["P"], ref["x"], per_solution=True)
e = errors(cppgd, ref["x"], ref["mu"], w_ref, s)

error = pd.DataFrame({"n": range(len(e)), "e": e.tolist()})
error.to_csv(out / "error.csv", index=False)
pd.DataFrame({"x": ref["x"], "mu": ref["mu"], "s": s}).to_csv(out / "signs.csv", index=False)
print(f"s_i = -1 at {int((s < 0).sum())} of {len(s)} points")
error.set_index("n").T

### Diagramme de bifurcation w(1/4) – μ

w_PGD(1/4, μ) sur 201 μ, avec λ_min en chaque μ (croix rouges : cols), sur les branches exactes de P = 2200 (C1, 100 éléments ; μ_cr de C1).

In [ ]:
mu_grid = torch.linspace(*CONFIG["mu_range"], 201)
x_q, mu_m = sample_monoms(cppgd, torch.tensor([0.25]), mu_grid)
lam = lambda_min(cppgd, mu_grid, CONFIG["P"], CONFIG["n_elements"], quadrature=QUAD)
diagram = pd.DataFrame({"mu": mu_grid, "w_quarter": (x_q * mu_m).sum(dim=0), "lambda_min": lam})
diagram.to_csv(out / "diagram.csv", index=False)
print(f"lambda_min <= 0 at {int((diagram.lambda_min <= 0).sum())} of {len(diagram)} mu")

In [ ]:
branches = pd.read_csv(C1 / "branches.csv")
mu_cr = pd.read_csv(C1 / "critical.csv").set_index("P").mu_cr[CONFIG["P"]]
plot_branches_k(
    branches, None, CONFIG["P"], mu_critical=mu_cr, series={f"PGD {RUN}": diagram},
    title=f"PGD {RUN}, P = {CONFIG['P']:g}: w(1/4) against mu (red crosses: lambda_min <= 0)",
).show()

### Exploration

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogy(error.n, error.e, "o-")
ax.set(xlabel="n (modes)", ylabel="e(n)", title=f"PGD {RUN}: relative L2 error on the LHS points")
ax.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
plot_loss_history(history.loss.tolist(), history["mode"].tolist()).show()

In [ ]:
x_fine = torch.linspace(0.0, 1.0, 201)
x_m, mu_m = sample_monoms(cppgd, x_fine, mu_grid)
plot_monoms(x_fine, x_m, mu_grid, mu_m).show()  # right panel: mu monoms (axis titled P by plot_monoms)

In [ ]:
# Deformed beam (x_hat + u_hat, w_hat): u at the nodes, linearly interpolated in x as in `main`
x_nodes = problem["x_nodes"]
u_nodes = axial_displacement(problem["field_layout"], cppgd, x_nodes, mu_m, CONFIG["P"])
u_fine = torch.from_numpy(np.stack(
    [np.interp(x_fine.numpy(), x_nodes.numpy(), u) for u in u_nodes.T.numpy()], axis=1,
))
x_hat, w_hat, u_hat, _ = rescale(x_fine, cppgd.assemble([x_fine, mu_grid]).detach(), u_fine,
                                 CONFIG["P"], **PHYS)
plot_deflection_slider(x_hat[:, None] + u_hat, mu_grid, w_hat,
                       title=f"PGD {RUN}: deformed beam (x + u, w) [m]").show()

## Run 4 - 20 elems LBFGS

Paramètres du « Choix initial » de l'Exp (V0). Pour un autre run : changer `RUN` et un seul facteur.

In [ ]:
RUN = "V3"
CONFIG = dict(
    P=2200.0, n_elements=50, n_mu=20, mu_range=(0.0, 40.0), n_modes=15,
    noise_amplitude=1e-2, seed=0,
    # Algorithm 1 of FENNI-PGD (values of Training_NeuROM on Develop_Daby)
    adam_lr=1e-2, adam_max_iter=5000, eta_c=1e-4, patience=10, no_improvement=None,
    min_useful=1,
    # final LBFGS on the whole decomposition (§3.2.2)
    lbfgs_lr=1.0, final_lbfgs_max_iter=500, tol=1e-8,
)
RECOMPUTE = True  # False: reload results/pgd/<RUN>/ when it exists
PROGRESS_EVERY = {"adam": 500, "lbfgs": 0.1}  # adam: every 500 it; lbfgs: every 10 % of its max_iter

In [ ]:
out = RESULTS / "pgd" / RUN
if RECOMPUTE or not (out / "decomposition.pt").exists():
    out.mkdir(parents=True, exist_ok=True)
    c = CONFIG
    lbfgs = Phase(
        lambda params: torch.optim.LBFGS(params, lr=c["lbfgs_lr"], line_search_fn="strong_wolfe"),
        Stagnation(tol=c["eta_c"], patience=c["patience"], no_improvement=None,
                   max_iter=c["final_lbfgs_max_iter"]),
        "lbfgs",
    )
    results = main(
        n_elements=c["n_elements"], n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"],
        noise_amplitude=c["noise_amplitude"], seed=c["seed"], phases=[lbfgs],
        enrichment_criterion=AnyOf(Usefulness(tol=c["eta_c"], min_useful=c["min_useful"]),
                                   MaxStages(c["n_modes"])),
        quadrature=QUAD, **PHYS, trainer=partial(SimultaneousTrainer, keep_optimizer = False, final_phases=[]),
        progress=ProgressLog(every=PROGRESS_EVERY),
    )
    cppgd, history, problem = results["cppgd"], results["history"], results["problem"]
    records = history.stages + ([history.final] if history.final is not None else [])
    save_decomposition(cppgd, out / "decomposition.pt", n_elements=c["n_elements"],
                       n_mu=c["n_mu"], mu_range=c["mu_range"], P=c["P"])
    (out / "config.json").write_text(json.dumps(CONFIG, indent=2))
    pd.DataFrame({
        "loss": history.losses,
        "mode": [r.stage for r in records for _ in r.losses],  # final LBFGS: stage = number of modes
        "phase": [p for r in records for p in r.phases],
    }).to_csv(out / "history.csv", index=False)
    pd.DataFrame([
        {"mode": r.stage, "n_iter": r.n_iter, "stop_reason": r.stop_reason, "energy": r.energy,
         "final": r is history.final}
        for r in records
    ]).to_csv(out / "stages.csv", index=False)
else:
    cppgd, _, problem = load_decomposition(out / "decomposition.pt", quadrature=QUAD)
    CONFIG = json.loads((out / "config.json").read_text())

history = pd.read_csv(out / "history.csv")
stages = pd.read_csv(out / "stages.csv")
stages

### Erreur e(n) sur les points LHS

Paragraphe « Erreur » de l'Exp ; s_i par q_PGDᵀ M q_EF (même maillage) ou grille fine (maillages différents).

In [ ]:
ref = torch.load(REFERENCE)
assert ref["P"] == CONFIG["P"] and tuple(ref["mu_box"]) == tuple(CONFIG["mu_range"]), "reference at another P or box"
if ref["n_elements"] != CONFIG["n_elements"]:
    print(f"warning: reference on {ref['n_elements']} elements in x, PGD on {CONFIG['n_elements']}")

s = align_signs(cppgd, ref["Q"], ref["mu"], ref["n_elements"], CONFIG["P"], M=mass_matrix(ref["n_elements"]))
w_ref = reference_values(ref["Q"], ref["n_elements"], CONFIG["P"], ref["x"], per_solution=True)
e = errors(cppgd, ref["x"], ref["mu"], w_ref, s)

error = pd.DataFrame({"n": range(len(e)), "e": e.tolist()})
error.to_csv(out / "error.csv", index=False)
pd.DataFrame({"x": ref["x"], "mu": ref["mu"], "s": s}).to_csv(out / "signs.csv", index=False)
print(f"s_i = -1 at {int((s < 0).sum())} of {len(s)} points")
error.set_index("n").T

### Diagramme de bifurcation w(1/4) – μ

w_PGD(1/4, μ) sur 201 μ, avec λ_min en chaque μ (croix rouges : cols), sur les branches exactes de P = 2200 (C1, 100 éléments ; μ_cr de C1).

In [ ]:
mu_grid = torch.linspace(*CONFIG["mu_range"], 201)
x_q, mu_m = sample_monoms(cppgd, torch.tensor([0.25]), mu_grid)
lam = lambda_min(cppgd, mu_grid, CONFIG["P"], CONFIG["n_elements"], quadrature=QUAD)
diagram = pd.DataFrame({"mu": mu_grid, "w_quarter": (x_q * mu_m).sum(dim=0), "lambda_min": lam})
diagram.to_csv(out / "diagram.csv", index=False)
print(f"lambda_min <= 0 at {int((diagram.lambda_min <= 0).sum())} of {len(diagram)} mu")

In [ ]:
branches = pd.read_csv(C1 / "branches.csv")
mu_cr = pd.read_csv(C1 / "critical.csv").set_index("P").mu_cr[CONFIG["P"]]
plot_branches_k(
    branches, None, CONFIG["P"], mu_critical=mu_cr, series={f"PGD {RUN}": diagram},
    title=f"PGD {RUN}, P = {CONFIG['P']:g}: w(1/4) against mu (red crosses: lambda_min <= 0)",
).show()

### Exploration

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogy(error.n, error.e, "o-")
ax.set(xlabel="n (modes)", ylabel="e(n)", title=f"PGD {RUN}: relative L2 error on the LHS points")
ax.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
plot_loss_history(history.loss.tolist(), history["mode"].tolist()).show()

In [ ]:
x_fine = torch.linspace(0.0, 1.0, 201)
x_m, mu_m = sample_monoms(cppgd, x_fine, mu_grid)
plot_monoms(x_fine, x_m, mu_grid, mu_m).show()  # right panel: mu monoms (axis titled P by plot_monoms)

In [ ]:
# Deformed beam (x_hat + u_hat, w_hat): u at the nodes, linearly interpolated in x as in `main`
x_nodes = problem["x_nodes"]
u_nodes = axial_displacement(problem["field_layout"], cppgd, x_nodes, mu_m, CONFIG["P"])
u_fine = torch.from_numpy(np.stack(
    [np.interp(x_fine.numpy(), x_nodes.numpy(), u) for u in u_nodes.T.numpy()], axis=1,
))
x_hat, w_hat, u_hat, _ = rescale(x_fine, cppgd.assemble([x_fine, mu_grid]).detach(), u_fine,
                                 CONFIG["P"], **PHYS)
plot_deflection_slider(x_hat[:, None] + u_hat, mu_grid, w_hat,
                       title=f"PGD {RUN}: deformed beam (x + u, w) [m]").show()